## Assignment 1 - Building a chatbot - AML 3304

Aarjeyan Shrestha - 18th Feb 2025

### Implementing a single neuron

In [ ]:
import numpy as np

# Define the sigmoid activation function and its derivative
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

# Softmax activation for the output layer
def softmax(z):
    # Subtract max for numerical stability
    exp_z = np.exp(z - np.max(z, axis=0, keepdims=True))
    return exp_z / np.sum(exp_z, axis=0, keepdims=True)

# Define the Neuron class
class Neuron:
    def __init__(self, input_size):
        # Initialize weights and bias with small random values
        self.weights = np.random.randn(input_size)
        self.bias = np.random.randn()

    def forward(self, inputs):
        # Compute the weighted sum (z)
        z = np.dot(inputs, self.weights) + self.bias
        # Apply the activation function
        return sigmoid(z)

# Example usage:
neuron = Neuron(input_size=3)
example_input = np.array([0.5, -1.2, 0.3])
output = neuron.forward(example_input)
print("Neuron output:", output)


Neuron output: 0.6575889443333276


### Building a multilayer neural network

In [2]:
class NeuralNetwork:
    def __init__(self, layer_sizes):
        """
        layer_sizes: a list of neuron counts for each layer.
        For example: [input_size, hidden_size, output_size]
        """
        self.num_layers = len(layer_sizes)
        self.weights = []
        self.biases = []
        # Initialize weights and biases for each layer transition
        for i in range(self.num_layers - 1):
            # Weight matrix shape: (next_layer_size, current_layer_size)
            w = np.random.randn(layer_sizes[i+1], layer_sizes[i])
            b = np.random.randn(layer_sizes[i+1], 1)
            self.weights.append(w)
            self.biases.append(b)

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def sigmoid_derivative(self, z):
        s = self.sigmoid(z)
        return s * (1 - s)

    def forward(self, x):
        """
        x: input vector of shape (input_size, 1)
        Returns:
         - activations: list of activations for each layer
         - zs: list of z vectors (weighted sums) for each layer
        """
        activation = x
        activations = [x]  # Store activations layer by layer
        zs = []  # Store z vectors layer by layer
        for w, b in zip(self.weights, self.biases):
            z = np.dot(w, activation) + b
            zs.append(z)
            activation = self.sigmoid(z)
            activations.append(activation)
        return activations, zs

    def compute_loss(self, output, target):
        # Mean Squared Error (MSE) Loss
        return np.mean((output - target) ** 2)

    def backward(self, activations, zs, target):
        """
        Backpropagation to compute gradients.
        Returns gradients for weights and biases.
        """
        # Initialize gradient lists with zeros matching the weights and biases shapes
        grad_w = [np.zeros(w.shape) for w in self.weights]
        grad_b = [np.zeros(b.shape) for b in self.biases]

        # Compute the error at the output layer
        delta = (activations[-1] - target) * self.sigmoid_derivative(zs[-1])
        grad_w[-1] = np.dot(delta, activations[-2].T)
        grad_b[-1] = delta

        # Propagate error backwards through the network
        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = self.sigmoid_derivative(z)
            delta = np.dot(self.weights[-l+1].T, delta) * sp
            grad_w[-l] = np.dot(delta, activations[-l-1].T)
            grad_b[-l] = delta

        return grad_w, grad_b

    def update_parameters(self, grad_w, grad_b, learning_rate):
        # Update weights and biases using gradient descent
        self.weights = [w - learning_rate * gw for w, gw in zip(self.weights, grad_w)]
        self.biases = [b - learning_rate * gb for b, gb in zip(self.biases, grad_b)]

    def train(self, x, target, learning_rate=0.1):
        """
        Train the network on a single example (x, target).
        """
        activations, zs = self.forward(x)
        grad_w, grad_b = self.backward(activations, zs, target)
        self.update_parameters(grad_w, grad_b, learning_rate)
        loss = self.compute_loss(activations[-1], target)
        return loss

# Example usage:
nn = NeuralNetwork(layer_sizes=[4, 5, 3])
x_example = np.random.randn(4, 1)
target_example = np.random.randn(3, 1)
loss = nn.train(x_example, target_example, learning_rate=0.05)
print("Loss after one training step:", loss)


Loss after one training step: 1.238199640790954


### Preparing training data

In [3]:
import re

# Define a small intents dataset
intents = {
    "greeting": {
         "patterns": ["Hi", "Hello", "Hey", "Good morning"],
         "responses": ["Hello!", "Hi there!", "Greetings!"]
    },
    "farewell": {
         "patterns": ["Bye", "See you", "Goodbye", "Later"],
         "responses": ["Goodbye!", "See you later!", "Farewell!"]
    },
    "thanks": {
         "patterns": ["Thanks", "Thank you", "Much appreciated"],
         "responses": ["You're welcome!", "No problem!", "Anytime!"]
    }
}

# Function to tokenize sentences (convert to lowercase and split words)
def tokenize(sentence):
    return re.findall(r'\w+', sentence.lower())

# Build the vocabulary (unique words across all patterns)
vocabulary = set()
for intent in intents.values():
    for pattern in intent["patterns"]:
        tokens = tokenize(pattern)
        vocabulary.update(tokens)
vocabulary = sorted(list(vocabulary))
print("Vocabulary:", vocabulary)

# Convert a sentence to a bag-of-words vector
def sentence_to_bow(sentence, vocabulary):
    tokens = tokenize(sentence)
    vector = np.array([1 if word in tokens else 0 for word in vocabulary])
    return vector.reshape(-1, 1)  # column vector

# Create intent-to-index mapping for one-hot encoding of labels
intents_list = list(intents.keys())
intent_to_index = {intent: i for i, intent in enumerate(intents_list)}
print("Intent mapping:", intent_to_index)

def intent_to_onehot(intent, intent_to_index):
    onehot = np.zeros(len(intent_to_index))
    onehot[intent_to_index[intent]] = 1
    return onehot.reshape(-1, 1)

# Create training data: each pattern is converted to a bag-of-words vector with a corresponding one-hot label.
training_data = []
for intent, data in intents.items():
    for pattern in data["patterns"]:
        bow = sentence_to_bow(pattern, vocabulary)
        label = intent_to_onehot(intent, intent_to_index)
        training_data.append((bow, label))

print("Number of training examples:", len(training_data))


Vocabulary: ['appreciated', 'bye', 'good', 'goodbye', 'hello', 'hey', 'hi', 'later', 'morning', 'much', 'see', 'thank', 'thanks', 'you']
Intent mapping: {'greeting': 0, 'farewell': 1, 'thanks': 2}
Number of training examples: 11


### Defining the network architecture and train

In [4]:
# Define network architecture
input_size = len(vocabulary)
hidden_size = 8
output_size = len(intents_list)

# Create the neural network
chatbot_nn = NeuralNetwork(layer_sizes=[input_size, hidden_size, output_size])

# Training parameters
epochs = 1000
learning_rate = 0.1
loss_history = []

# Training loop
for epoch in range(epochs):
    epoch_loss = 0
    for x, target in training_data:
        loss = chatbot_nn.train(x, target, learning_rate=learning_rate)
        epoch_loss += loss
    loss_history.append(epoch_loss / len(training_data))
    if (epoch + 1) % 100 == 0:
        print("Epoch {}: Loss = {:.4f}".format(epoch + 1, loss_history[-1]))


Epoch 100: Loss = 0.1470
Epoch 200: Loss = 0.0620
Epoch 300: Loss = 0.0212
Epoch 400: Loss = 0.0104
Epoch 500: Loss = 0.0066
Epoch 600: Loss = 0.0047
Epoch 700: Loss = 0.0036
Epoch 800: Loss = 0.0029
Epoch 900: Loss = 0.0024
Epoch 1000: Loss = 0.0021


### Building the chatbot interface

In [5]:
import random

def classify_intent(sentence, nn, vocabulary):
    """
    Convert the sentence to bag-of-words, run it through the network,
    and return the predicted intent along with the network's output probabilities.
    """
    bow = sentence_to_bow(sentence, vocabulary)
    activations, _ = nn.forward(bow)
    output = activations[-1]
    intent_index = np.argmax(output)
    return intents_list[intent_index], output

def chat():
    print("Chatbot is online! Type 'quit' to exit.")
    while True:
        user_input = input("You: ")
        if user_input.lower() == 'quit':
            print("Chatbot: Goodbye!")
            break
        intent, output_probs = classify_intent(user_input, chatbot_nn, vocabulary)
        responses = intents[intent]["responses"]
        response = random.choice(responses)
        print("Chatbot ({}): {}".format(intent, response))

# Start the chatbot
chat()


Chatbot is online! Type 'quit' to exit.
You: Hi
Chatbot (greeting): Hello!
You: ola
Chatbot (farewell): Goodbye!
You: how are you
Chatbot (farewell): See you later!
You: Thank you
Chatbot (thanks): No problem!
You: Bye
Chatbot (farewell): See you later!
You: quit
Chatbot: Goodbye!
